In [1]:
import pandas as pd

# 1. 극지 데이터 불러오기 ('../data/' 로 상위 폴더 경로 지정)
df_sejong = pd.read_csv('../data/openmetro_sejong.csv', parse_dates=['Date'])
df_jangbogo = pd.read_csv('../data/openmetro_jangbogo.csv', parse_dates=['Date'])

# 잘 불러와졌는지 상위 3줄 확인
print("=== 세종기지 데이터 ===")
display(df_sejong.head(3))



# 2. 기상청 대구/부산 데이터 불러오기 (한글 깨짐 방지 인코딩 추가)
df_daegu = pd.read_csv('../data/kma_daegu.csv', encoding='cp949')
df_busan = pd.read_csv('../data/kma_busan.csv', encoding='cp949')

# 컬럼명 확인 (병합을 위해 '날짜' 컬럼 이름을 확인해야 함)
print("=== 대구 기상청 원본 컬럼명 ===")
print(df_daegu.columns.tolist())

=== 세종기지 데이터 ===


,Date,Temp_Mean,Temp_Max,Temp_Min,Humidity_Mean,Wind_Mean
0,2016-01-01,1.2,1.8,0.5,95,28.0
1,2016-01-02,1.8,2.1,1.7,98,28.1
2,2016-01-03,1.4,2.5,-0.6,93,39.6


=== 대구 기상청 원본 컬럼명 ===
['지점', '지점명', '일시', '평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '평균 풍속(m/s)', '평균 상대습도(%)']


In [2]:
# 3. 기상청 데이터 컬럼명 변경 및 전처리
kma_rename_dict = {
    '일시': 'Date',
    '평균기온(°C)': 'Temp_Mean',
    '최고기온(°C)': 'Temp_Max',
    '최저기온(°C)': 'Temp_Min',
    '평균 상대습도(%)': 'Humidity_Mean',
    '평균 풍속(m/s)': 'Wind_Mean'
}

# 대구, 부산 데이터 컬럼명 변경 및 불필요한 컬럼(지점, 지점명) 제거
df_daegu = df_daegu.rename(columns=kma_rename_dict).drop(columns=['지점', '지점명'])
df_busan = df_busan.rename(columns=kma_rename_dict).drop(columns=['지점', '지점명'])

# Date 컬럼을 극지 데이터와 똑같은 날짜(datetime) 형식으로 통일
df_daegu['Date'] = pd.to_datetime(df_daegu['Date'])
df_busan['Date'] = pd.to_datetime(df_busan['Date'])

# 4. 나중에 헷갈리지 않게 컬럼명에 지역 이름(Prefix) 붙이기
df_daegu.columns = ['Date'] + [f'Daegu_{col}' for col in df_daegu.columns if col != 'Date']
df_busan.columns = ['Date'] + [f'Busan_{col}' for col in df_busan.columns if col != 'Date']
df_sejong.columns = ['Date'] + [f'Sejong_{col}' for col in df_sejong.columns if col != 'Date']
df_jangbogo.columns = ['Date'] + [f'Jangbogo_{col}' for col in df_jangbogo.columns if col != 'Date']

# 5. 하나의 거대한 데이터프레임으로 병합! (Date 기준 교집합 교차)
df_merged = pd.merge(df_sejong, df_jangbogo, on='Date', how='inner')
df_merged = pd.merge(df_merged, df_daegu, on='Date', how='inner')
df_merged = pd.merge(df_merged, df_busan, on='Date', how='inner')

print("=== 최종 병합된 4개 지역 데이터 상위 3줄 ===")
display(df_merged.head(3))

=== 최종 병합된 4개 지역 데이터 상위 3줄 ===


,Date,Sejong_Temp_Mean,Sejong_Temp_Max,Sejong_Temp_Min,Sejong_Humidity_Mean,Sejong_Wind_Mean,Jangbogo_Temp_Mean,Jangbogo_Temp_Max,Jangbogo_Temp_Min,Jangbogo_Humidity_Mean,...,Daegu_Temp_Mean,Daegu_Temp_Min,Daegu_Temp_Max,Daegu_Wind_Mean,Daegu_Humidity_Mean,Busan_Temp_Mean,Busan_Temp_Min,Busan_Temp_Max,Busan_Wind_Mean,Busan_Humidity_Mean
0,2016-01-01,1.2,1.8,0.5,95,28.0,-7.1,-5.4,-8.5,58,...,1.7,-4.0,8.0,1.0,70.5,5.3,1.1,10.9,4.3,52.1
1,2016-01-02,1.8,2.1,1.7,98,28.1,-8.8,-7.1,-10.3,48,...,3.0,-2.7,9.9,0.7,74.8,8.1,4.6,12.2,4.0,62.9
2,2016-01-03,1.4,2.5,-0.6,93,39.6,-7.4,-5.6,-9.7,40,...,4.9,-2.3,13.9,0.7,77.0,11.4,7.9,16.6,2.7,68.3


# Moving Average

In [3]:
# 1. 결측치(NaN) 선형 보간법으로 채우기
# 기상 데이터는 앞뒤 날짜의 영향을 크게 받으므로 시계열에 적합한 선형 보간(linear) 사용
df_merged = df_merged.interpolate(method='linear')

# 2. 7일 이동평균(Smoothing) 적용
# 날짜(Date) 컬럼을 제외한 모든 수치형 데이터에 대해 7일 평균을 구함
numeric_cols = df_merged.columns.drop('Date')
df_smoothed = df_merged.copy()

# min_periods=1 옵션을 주면 앞쪽 1~6일차 데이터도 NaN이 되지 않고 가능한 일수만큼만 평균을 냄
df_smoothed[numeric_cols] = df_merged[numeric_cols].rolling(window=7, min_periods=1).mean()

print("=== 결측치 보간 및 7일 스무딩 처리 완료 ===")
display(df_smoothed.head(5))

=== 결측치 보간 및 7일 스무딩 처리 완료 ===


,Date,Sejong_Temp_Mean,Sejong_Temp_Max,Sejong_Temp_Min,Sejong_Humidity_Mean,Sejong_Wind_Mean,Jangbogo_Temp_Mean,Jangbogo_Temp_Max,Jangbogo_Temp_Min,Jangbogo_Humidity_Mean,...,Daegu_Temp_Mean,Daegu_Temp_Min,Daegu_Temp_Max,Daegu_Wind_Mean,Daegu_Humidity_Mean,Busan_Temp_Mean,Busan_Temp_Min,Busan_Temp_Max,Busan_Wind_Mean,Busan_Humidity_Mean
0,2016-01-01,1.200000,1.800000,0.500000,95.000000,28.00,-7.100000,-5.400000,-8.50,58.000000,...,1.70,-4.000,8.00,1.000,70.50,5.300000,1.100000,10.900000,4.300000,52.1
1,2016-01-02,1.500000,1.950000,1.100000,96.500000,28.05,-7.950000,-6.250000,-9.40,53.000000,...,2.35,-3.350,8.95,0.850,72.65,6.700000,2.850000,11.550000,4.150000,57.5
2,2016-01-03,1.466667,2.133333,0.533333,95.333333,31.90,-7.766667,-6.033333,-9.50,48.666667,...,3.20,-3.000,10.60,0.800,74.10,8.266667,4.533333,13.233333,3.666667,61.1
3,2016-01-04,1.000000,1.550000,0.100000,93.500000,33.50,-7.250000,-4.925000,-9.10,49.250000,...,3.85,-1.975,10.80,1.025,70.10,8.800000,5.300000,13.725000,3.425000,57.4
4,2016-01-05,0.640000,1.180000,-0.200000,92.000000,31.48,-6.900000,-4.620000,-8.84,52.400000,...,3.66,-1.400,9.82,1.300,61.96,8.100000,4.880000,12.600000,3.360000,54.1


In [4]:
# 3. 타겟 파생변수 생성: 여름철 불쾌지수 & 겨울철 체감온도

# 겨울철 한파 체감온도 함수 (최저기온, 풍속 기준)
# 기상청 공식: 13.12 + 0.6215*T - 11.37*V^0.16 + 0.3965*V^0.16*T (V는 km/h)
def calc_wind_chill(temp_min, wind_m_s):
    wind_kmh = wind_m_s * 3.6 # m/s를 km/h로 변환
    return 13.12 + 0.6215 * temp_min - 11.37 * (wind_kmh**0.16) + 0.3965 * (wind_kmh**0.16) * temp_min

# 대구, 부산 겨울철 체감온도 열 추가
df_smoothed['Daegu_WindChill_Min'] = calc_wind_chill(df_smoothed['Daegu_Temp_Min'], df_smoothed['Daegu_Wind_Mean'])
df_smoothed['Busan_WindChill_Min'] = calc_wind_chill(df_smoothed['Busan_Temp_Min'], df_smoothed['Busan_Wind_Mean'])

# 여름철 폭염 불쾌지수 함수 (최고기온, 상대습도 기준)
# 공식: 0.81 * T + 0.01 * H * (0.99 * T - 14.3) + 46.3
def calc_thi(temp_max, humidity):
    return 0.81 * temp_max + 0.01 * humidity * (0.99 * temp_max - 14.3) + 46.3

# 대구, 부산 여름철 불쾌지수 열 추가
df_smoothed['Daegu_THI_Max'] = calc_thi(df_smoothed['Daegu_Temp_Max'], df_smoothed['Daegu_Humidity_Mean'])
df_smoothed['Busan_THI_Max'] = calc_thi(df_smoothed['Busan_Temp_Max'], df_smoothed['Busan_Humidity_Mean'])

# 최종 전처리된 데이터를 새로운 CSV로 저장 (다음 분석 단계에서 바로 쓰기 위함)
df_smoothed.to_csv('../data/polar_weather_preprocessed.csv', index=False)

print("=== 여름철/겨울철 타겟 파생변수 생성 및 최종 저장 완료 ===")
display(df_smoothed[['Date', 'Daegu_WindChill_Min', 'Busan_WindChill_Min', 'Daegu_THI_Max', 'Busan_THI_Max']].head())

=== 여름철/겨울철 타겟 파생변수 생성 및 최종 저장 완료 ===


,Date,Daegu_WindChill_Min,Busan_WindChill_Min,Daegu_THI_Max,Busan_THI_Max
0,2016-01-01,-5.269010,-3.145053,48.282100,53.300811
1,2016-01-02,-4.148611,-0.891923,49.597703,54.007837
2,2016-01-03,-3.620122,1.472457,52.065754,56.286411
3,2016-01-04,-3.083980,2.560246,52.518792,57.008418
4,2016-01-05,-3.015260,2.093472,51.417547,55.518134


In [6]:
# 4. 탈계절화 (Deseasonalization)
df_smoothed['DayOfYear'] = df_smoothed['Date'].dt.dayofyear

# 🚨 기존 타겟에 남극 풍속(Wind_Mean) 2개를 추가!
anomaly_targets = [
    'Sejong_Temp_Mean', 'Jangbogo_Temp_Mean',
    'Sejong_Wind_Mean', 'Jangbogo_Wind_Mean',  # 👈 새로 추가된 부분
    'Daegu_WindChill_Min', 'Busan_WindChill_Min',
    'Daegu_THI_Max', 'Busan_THI_Max'
]

for col in anomaly_targets:
    climatology = df_smoothed.groupby('DayOfYear')[col].transform('mean')
    df_smoothed[f'{col}_Anomaly'] = df_smoothed[col] - climatology

print("=== 탈계절화(Anomaly) 변수 생성 완료 (풍속 포함) ===")
df_smoothed.to_csv('../data/polar_weather_preprocessed.csv', index=False)

=== 탈계절화(Anomaly) 변수 생성 완료 (풍속 포함) ===
